# 关于中值滤波和均值滤波的一个简单例子

以简单矩阵为例，描述中值滤波、均值滤波的实际计算结果。

对于矩阵$A$、$B$，分别计算中值滤波和均值滤波。

$$
A = \begin{bmatrix}
   5 & 9 & 7 \\
   3 & 3 & 0 \\
   1 & 6 & 2
  \end{bmatrix}
$$

$$
B = \begin{bmatrix}
   5 & 9 & 7 \\
   3 & 12 & 0 \\
   1 & 6 & 2
  \end{bmatrix}  
$$

## 中值滤波

中值滤波是一种非线性滤波技术，它通过取邻域内像素值的中位数来替代中心像素值，从而有效地去除噪声。对于矩阵$A$和$B$，我们可以使用一个3x3的窗口来进行中值滤波。

In [1]:
import cv2
import numpy as np

img_a = np.array([[5, 9, 7],
                  [3, 3, 0],
                    [1, 6, 2]], dtype=np.uint8)

img_b = np.array([[5, 9, 7],
                  [3, 12, 0],
                  [1, 6, 2]], dtype=np.uint8)

blurred_a = cv2.medianBlur(img_a, 3)
blurred_b = cv2.medianBlur(img_b, 3)

img_a, blurred_a, img_b, blurred_b

(array([[5, 9, 7],
        [3, 3, 0],
        [1, 6, 2]], dtype=uint8),
 array([[5, 5, 7],
        [3, 3, 3],
        [3, 2, 2]], dtype=uint8),
 array([[ 5,  9,  7],
        [ 3, 12,  0],
        [ 1,  6,  2]], dtype=uint8),
 array([[5, 7, 7],
        [5, 5, 6],
        [3, 2, 2]], dtype=uint8))

## 均值滤波


In [2]:
mean_blurred_a = cv2.blur(img_a, (3, 3))
mean_blurred_b = cv2.blur(img_b, (3, 3))

img_a, mean_blurred_a, img_b, mean_blurred_b

(array([[5, 9, 7],
        [3, 3, 0],
        [1, 6, 2]], dtype=uint8),
 array([[5, 4, 4],
        [5, 4, 5],
        [3, 2, 3]], dtype=uint8),
 array([[ 5,  9,  7],
        [ 3, 12,  0],
        [ 1,  6,  2]], dtype=uint8),
 array([[9, 6, 8],
        [7, 5, 7],
        [7, 4, 7]], dtype=uint8))

In [3]:
border_replicate = cv2.blur(img_a, (3, 3), borderType=cv2.BORDER_REPLICATE)

In [4]:
border_reflect = cv2.blur(img_a, (3, 3), borderType=cv2.BORDER_REFLECT)
border_reflect_101 = cv2.blur(img_a, (3, 3), borderType=cv2.BORDER_REFLECT_101)
border_constant = cv2.blur(img_a, (3, 3), borderType=cv2.BORDER_CONSTANT)
border_isolated = cv2.blur(img_a, (3, 3), borderType=cv2.BORDER_ISOLATED)
border_default = cv2.blur(img_a, (3, 3), borderType=cv2.BORDER_DEFAULT)

border_replicate, border_reflect, border_reflect_101, border_constant, border_isolated, border_default

(array([[5, 5, 5],
        [4, 4, 4],
        [3, 3, 3]], dtype=uint8),
 array([[5, 5, 5],
        [4, 4, 4],
        [3, 3, 3]], dtype=uint8),
 array([[5, 4, 4],
        [5, 4, 5],
        [3, 2, 3]], dtype=uint8),
 array([[2, 3, 2],
        [3, 4, 3],
        [1, 2, 1]], dtype=uint8),
 array([[2, 3, 2],
        [3, 4, 3],
        [1, 2, 1]], dtype=uint8),
 array([[5, 4, 4],
        [5, 4, 5],
        [3, 2, 3]], dtype=uint8))

# OpenCV BorderType 边界填充类型解析

`BorderType` 是OpenCV图像做边界扩充（填充）时的枚举类型，常见于 `cv2.copyMakeBorder()`、滤波函数（`blur`、`GaussianBlur`等内部边界处理）。

> 
> 函数原型：
> `dst = cv2.copyMakeBorder(src, top, bottom, left, right, borderType[, value])`

| BorderType常量 | 数值 | 名称 | 原理说明 | 示例(以一维序列`[a b c d]`向左右扩充2像素举例) |
| --- | --- | --- | --- | --- |
| cv2.BORDER_CONSTANT | 0 | 常量填充 | 使用给定固定常量`value`填充边界，全部填充同一个颜色值。 | `[v v a b c d v v]` |
| cv2.BORDER_REPLICATE | 1 | 复制填充（复制边缘） | 直接复制图像最边缘那一行/列像素向外延展。 | `[a a a b c d d d]` |
| cv2.BORDER_REFLECT | 2 | 反射填充（镜像，不含原边界点） | 以图像边缘为对称轴做镜像反射，**边界像素不重复**。   镜像：`a b c d`，反射：`b a ‖ a b c d ‖ d c` | `[b a a b c d d c]` |
| cv2.BORDER_WRAP | 3 | 环绕填充/周期平铺 | 把图像当作周期循环，用图像对侧的像素来填充边界，类似平铺复制整张图。   相当于循环卷绕。原序列`a b c d`，左边取末尾，右边取开头。 | `[c d a b c d a b]` |
| cv2.BORDER_REFLECT_101   别名 BORDER_DEFAULT | 4 | 反射101（默认反射，包含边界点） | 最常用！以边缘像素为轴镜像，**边缘像素只出现一次**，OpenCV滤波函数**内部默认边界模式就是BORDER_DEFAULT**。   镜像：`a b c d`，反射：`c b a ‖ a b c d ‖ d c b` | `[c b a a b c d d c b]` |

---

## 详细逐条解析

### 1. BORDER_CONSTANT (0) 常量填充

- 原理：边界全部填充用户指定的颜色`value`，可以是灰度值或者BGR三元组。
- 适用：需要明确给图像增加纯色边框，例如给图片加黑色/白色边框。
- 注意：做图像滤波时一般不推荐，引入人为的常量边缘，会在图像边界产生伪影。

```
import cv2
import numpy as np
img = np.array([[1,2,3],[4,5,6],[7,8,9]],dtype=np.uint8)
res = cv2.copyMakeBorder(img,1,1,1,1,cv2.BORDER_CONSTANT,value=(0,))
```

### 2. BORDER_REPLICATE (1) 复制边缘填充

- 原理：直接重复图像最外侧行、列像素向外扩展。
- 特点：实现简单；边界梯度平缓，适合部分实景图像处理。
- 缺点：边缘像素会被大量重复，图像边界特征被过度放大。

### 3. BORDER_REFLECT (2) 反射（不含边界点镜像）

- 镜像时，**边缘像素不参与镜像对称轴，相邻两个像素翻转**
序列：`[a b c d]`向左扩2：取`b a`，向右扩2取`d c`
- 注意：和`BORDER_REFLECT_101`容易混淆，日常滤波很少使用。

### 4. BORDER_WRAP (3) 环绕填充

- 原理：周期循环，就像把图像左右、上下拼接在一起。左边边界拿图像最右侧像素，右边拿图像最左侧像素。
- 适用：周期性纹理图像，比如重复纹理、频谱图；**普通实拍照片不要用**，会把对侧内容复制过来造成错误。

### 5. BORDER_REFLECT_101 (4) 别名 BORDER_DEFAULT，OpenCV默认模式

> 
> `cv2.BORDER_DEFAULT` 等价于 `cv2.BORDER_REFLECT_101`

- 原理：以边界像素作为镜像轴做反射，边界像素只保留1次。
序列`a b c d`，向左扩充两个像素：`c b`；向右扩充两个像素：`c b`。
- 最常用：`cv2.GaussianBlur`、`cv2.Canny`等绝大多数OpenCV滤波函数内部**默认使用该模式**。
- 优点：不会引入突变、不会引入新的人为像素，边界过渡平滑，是绝大多数图像任务首选。

## 一维直观对比演示

原一维信号：`[10, 20, 30, 40]`，左右各向外填充2个点

1. BORDER_CONSTANT(value=0) → `[0, 0, 10, 20, 30, 40, 0, 0]`
2. BORDER_REPLICATE → `[10,10,10,20,30,40,40,40]`
3. BORDER_REFLECT → `[20,10,10,20,30,40,40,30]`
4. BORDER_WRAP → `[30,40,10,20,30,40,10,20]`
5. BORDER_REFLECT_101(DEFAULT) → `[30,20,10,20,30,40,30,20]`

## 选型总结（工程/机器视觉）

1. 普通图像滤波、模糊、梯度算子：优先 **BORDER_DEFAULT（REFLECT_101）**，OpenCV内置默认。
2. 需要增加纯色边框：选 **BORDER_CONSTANT**。
3. 周期性纹理、周期仿真图像：选 **BORDER_WRAP**。
4. BORDER_REFLECT 很少使用；BORDER_REPLICate部分场景使用。

## 完整可运行测试代码

```
import cv2
import numpy as np

src = np.array([[10, 20, 30],
                [40, 50, 60],
                [70, 80, 90]], dtype=np.uint8)

top,bottom,left,right = 2,2,2,2

res_const = cv2.copyMakeBorder(src,top,bottom,left,right,cv2.BORDER_CONSTANT,value=0)
res_rep = cv2.copyMakeBorder(src,top,bottom,left,right,cv2.BORDER_REPLICATE)
res_reflect = cv2.copyMakeBorder(src,top,bottom,left,right,cv2.BORDER_REFLECT)
res_wrap = cv2.copyMakeBorder(src,top,bottom,left,right,cv2.BORDER_WRAP)
res_ref101 = cv2.copyMakeBorder(src,top,bottom,left,right,cv2.BORDER_DEFAULT)

print("CONSTANT:\n",res_const)
print("\nREPLICATE:\n",res_rep)
print("\nREFLECT:\n",res_reflect)
print("\nWRAP:\n",res_wrap)
print("\nREFLECT_101(DEFAULT):\n",res_ref101)
```
